# trt_yolo_bench - Colab Runner

This notebook runs the repo pipeline in Colab from clone to scoring.
Run cells top-to-bottom.

## 1) Runtime checks
In Colab: Runtime -> Change runtime type -> GPU (T4 recommended).

In [ ]:
!nvidia-smi
!nvcc --version

## 2) Clone or enter repo

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/chao-dotcom/trt_yolo_bench.git"
REPO_DIR = Path("/content/trt_yolo_bench")

if not REPO_DIR.exists():
    !git clone {REPO_URL} /content/trt_yolo_bench
else:
    %cd /content/trt_yolo_bench
    !git pull --ff-only

%cd /content/trt_yolo_bench
!git status --short

## 3) Install Python dependencies

In [ ]:
!python -m pip install -q --upgrade pip
!pip install -q ultralytics onnx pycocotools requests opencv-python numpy

# TensorRT / CUDA deps.
# Pinned to the last TensorRT 10.x release: TensorRT 11.0 removed the implicit
# INT8 calibrator API (IInt8EntropyCalibrator2) that calibrator.py depends on.
!pip install -q tensorrt-cu12==10.16.1.11 || pip install -q tensorrt==10.16.1.11
!pip install -q pycuda

# OpenCV C++ dev headers/cmake config for the harness build (opencv-python above
# only ships its own bundled runtime libs, not the dev package find_package needs).
!apt-get -qq update && apt-get -qq install -y libopencv-dev

# Fail fast here instead of deep inside calibrator.py at step 6 if a future
# Colab image silently resolves a newer/incompatible TensorRT build.
import tensorrt as trt

assert hasattr(trt, "IInt8EntropyCalibrator2"), (
    f"tensorrt {trt.__version__} is missing IInt8EntropyCalibrator2 "
    "(removed in TensorRT 11+). Re-pin the install above to a 10.x release."
)
print(f"[ok] tensorrt {trt.__version__} has IInt8EntropyCalibrator2")

## 4) Prepare deterministic COCO slices + images

In [ ]:
!python export/download_coco_slice.py

## 5) Export ONNX

In [ ]:
!python export/export_onnx.py --weights yolo11n.pt --out-dir models

## 6) Build TensorRT engines (FP32/FP16/INT8)

In [ ]:
!python export/build_engines.py --onnx models/yolo11n.onnx --engines-dir engines

## 7) Build C++ harness

In [ ]:
import glob
import os
import subprocess
import sysconfig

os.chdir("/content/trt_yolo_bench/harness")

# The pip-installed tensorrt-cu12 wheel ships the runtime .so files but not the
# C++ headers (NvInfer.h etc). Fetch just the matching headers from NVIDIA's OSS
# repo (v10.16 matches the pinned tensorrt-cu12==10.16.1.11 from step 3) so the
# engine-building step and this harness link against the identical TensorRT build.
if not os.path.isdir("/content/tensorrt_oss/include"):
    subprocess.run(
        [
            "git", "clone", "--branch", "v10.16", "--depth", "1",
            "--filter=blob:none", "--sparse",
            "https://github.com/NVIDIA/TensorRT.git", "/content/tensorrt_oss",
        ],
        check=True,
    )
    subprocess.run(
        ["git", "-C", "/content/tensorrt_oss", "sparse-checkout", "set", "include"],
        check=True,
    )


def find_lib(pattern: str) -> str:
    site = sysconfig.get_paths()["purelib"]
    matches = sorted(glob.glob(os.path.join(site, "**", pattern), recursive=True))
    if not matches:
        raise FileNotFoundError(f"{pattern} not found under {site}")
    return matches[-1]


trt_lib = find_lib("libnvinfer.so*")
trt_lib_dir = os.path.dirname(trt_lib)
print(f"TensorRT include: /content/tensorrt_oss/include")
print(f"TensorRT lib: {trt_lib}")

subprocess.run(
    [
        "cmake", "-S", ".", "-B", "build",
        "-DTENSORRT_INCLUDE_DIR=/content/tensorrt_oss/include",
        f"-DTENSORRT_LIB={trt_lib}",
        f"-DCMAKE_EXE_LINKER_FLAGS=-Wl,-rpath,{trt_lib_dir}",
    ],
    check=True,
)
subprocess.run(["cmake", "--build", "build", "-j"], check=True)

os.chdir("/content/trt_yolo_bench")

## 8) Run harness (start with FP16)

In [ ]:
!./harness/build/trt_yolo_bench engines/yolo11n_fp16.engine data/val_slice.json --out results_fp16.json

## 9) Score COCO metrics

In [ ]:
!python eval/score_coco.py --dets results_fp16.json

## 10) Optional: run FP32 and INT8
Uncomment and run if engine files were built successfully.

In [ ]:
# !./harness/build/trt_yolo_bench engines/yolo11n_fp32.engine data/val_slice.json --out results_fp32.json
# !python eval/score_coco.py --dets results_fp32.json
# !./harness/build/trt_yolo_bench engines/yolo11n_int8.engine data/val_slice.json --out results_int8.json
# !python eval/score_coco.py --dets results_int8.json